# Introduction to Bayesian Inference

## 1. Title and Overview

Welcome to Bayesian Inference. You have likely spent significant time mastering the Frequentist approach (p-values, hypothesis tests, and confidence intervals). In the Frequentist framework, we treat the true parameter as a fixed, objective constant, focusing on the long-run frequency of our data.

Bayesian Inference represents a philosophical and mathematical shift. It treats uncertainty not as a frequency, but as a degree of belief. We begin with an initial belief, gather evidence, and update our belief. This notebook will guide you through the mathematical engine of belief updating, conjugate priors, and practical applications in A/B testing.

In [ ]:
# 2. Setup and Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set display options for clear output
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')

# Set random seed for complete reproducibility
np.random.seed(42)
print('Environment initialized successfully.')

## 3. The Philosophy: Belief vs. Frequency

The Frequentist View: 'The drug effectiveness is a fixed, unknown value. If I repeat this clinical trial 1,000 times, my confidence interval will capture the true value 95 percent of the time.'

The Bayesian View: 'I have an initial belief about the drug effectiveness based on past trials (the Prior). As I see new data from this trial (the Likelihood), I update my belief (the Posterior). My result is a probability distribution of the drug effectiveness.'

Bayes Theorem is the formalization of this learning process:
Posterior is proportional to Likelihood * Prior
P(Parameter | Data) = ( P(Data | Parameter) * P(Parameter) ) / P(Data)

## 4. Data Creation: Pharmaceutical Clinical Trial

Let us simulate a scenario where we are testing the efficacy of a new drug. The outcome is binary: a patient either recovers (1) or does not recover (0). We will conduct a small clinical trial.

In [ ]:
# Generate synthetic clinical trial data
n_patients = 20
true_efficacy = 0.70  # The hidden truth we are trying to estimate

# Simulate patient outcomes (1 = recovered, 0 = not recovered)
trial_data = np.random.binomial(n=1, p=true_efficacy, size=n_patients)

recoveries = np.sum(trial_data)
failures = n_patients - recoveries

print(f'Clinical Trial Results:')
print(f'Total Patients: {n_patients}')
print(f'Recoveries (Successes): {recoveries}')
print(f'Failures: {failures}')
print(f'Raw Success Rate: {recoveries / n_patients:.2f}')

## 5. Core Concept 1: Grid Approximation

To understand Bayesian updating, we can compute the Posterior distribution using a discrete grid. We evaluate our prior belief and the likelihood of the data at a finite number of points between 0 and 1.

1. Define a grid of possible parameter values (e.g., 0.00, 0.01, ..., 1.00).
2. Define the Prior probability for each grid value.
3. Compute the Likelihood of our observed data at each grid value.
4. Multiply Prior by Likelihood, then normalize to sum to 1 to get the Posterior.

In [ ]:
# 1. Define the grid (100 points between 0 and 1)
p_grid = np.linspace(0, 1, 100)

# 2. Define a Uniform Prior (we believe all efficacies are equally likely initially)
prior = np.ones(100) / 100

# 3. Compute Likelihood using the Binomial PMF
# Probability of getting exactly 'recoveries' out of 'n_patients' given parameter 'p'
likelihood = stats.binom.pmf(k=recoveries, n=n_patients, p=p_grid)

# 4. Compute unnormalized posterior, then normalize
unnormalized_posterior = likelihood * prior
posterior = unnormalized_posterior / np.sum(unnormalized_posterior)

print('Grid approximation complete.')
print(f'Maximum Posterior Probability occurs at p = {p_grid[np.argmax(posterior)]:.2f}')

## 6. Visualizing the Grid Approximation

Let us plot the Prior, Likelihood, and Posterior to see how our belief was updated by the data.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot Prior
axes[0].plot(p_grid, prior, color='blue', linewidth=2)
axes[0].set_title('Prior Distribution')
axes[0].set_xlabel('Probability of Efficacy')
axes[0].set_ylabel('Belief')
axes[0].set_ylim(0, max(posterior) * 1.1)

# Plot Likelihood
axes[1].plot(p_grid, likelihood, color='green', linewidth=2)
axes[1].set_title('Likelihood (Evidence)')
axes[1].set_xlabel('Probability of Efficacy')

# Plot Posterior
axes[2].plot(p_grid, posterior, color='purple', linewidth=2)
axes[2].set_title('Posterior Distribution')
axes[2].set_xlabel('Probability of Efficacy')

plt.tight_layout()
plt.show()
print('The flat prior is updated by the bell-shaped likelihood to form the posterior.')

## 7. Core Concept 2: Conjugate Priors (The Analytical Shortcut)

Grid approximation is great for intuition, but computationally expensive for complex models. When our Likelihood is Binomial, we can use a Beta distribution as our Prior. 

The Beta distribution is a 'Conjugate Prior' for the Binomial likelihood. This means the Posterior will also be a Beta distribution! The math becomes simple addition:

Prior: Beta(alpha_prior, beta_prior)
Data: successes, failures
Posterior: Beta(alpha_prior + successes, beta_prior + failures)

In [ ]:
# A Uniform prior is mathematically equivalent to Beta(1, 1)
alpha_prior = 1
beta_prior = 1

# Analytical Bayesian Update
alpha_posterior = alpha_prior + recoveries
beta_posterior = beta_prior + failures

print(f'Analytical Update Results:')
print(f'Prior: Beta({alpha_prior}, {beta_prior})')
print(f'Posterior: Beta({alpha_posterior}, {beta_posterior})')

# Calculate the mean of the Beta distribution: alpha / (alpha + beta)
posterior_mean = alpha_posterior / (alpha_posterior + beta_posterior)
print(f'Posterior Mean (Expected Efficacy): {posterior_mean:.3f}')

## 8. Visualizing Conjugate Priors

We can use scipy.stats to generate the continuous Beta distributions and visualize the exact analytical curves.

In [ ]:
x_vals = np.linspace(0, 1, 500)

prior_pdf = stats.beta.pdf(x_vals, alpha_prior, beta_prior)
posterior_pdf = stats.beta.pdf(x_vals, alpha_posterior, beta_posterior)

plt.figure(figsize=(10, 6))
plt.plot(x_vals, prior_pdf, 'b--', lw=2, label='Prior: Beta(1,1)')
plt.plot(x_vals, posterior_pdf, 'purple', lw=3, label=f'Posterior: Beta({alpha_posterior},{beta_posterior})')
plt.fill_between(x_vals, 0, posterior_pdf, color='purple', alpha=0.2)

plt.axvline(posterior_mean, color='black', linestyle=':', label=f'Posterior Mean: {posterior_mean:.2f}')
plt.title('Analytical Bayesian Updating with Conjugate Priors', fontsize=14)
plt.xlabel('Efficacy Probability (theta)', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 9. Core Concept 3: The Power of Priors

What if we had previous knowledge? Imagine a related drug previously showed a 50 percent efficacy, and we are quite confident our new drug will be similar. We can encode this as a 'Strong Prior', like Beta(20, 20).

Let us see how different priors react to the exact same dataset.

In [ ]:
# Scenario A: Weak/Flat Prior (Beta 1, 1)
alpha_weak_post = 1 + recoveries
beta_weak_post = 1 + failures

# Scenario B: Strong Prior centered at 0.50 (Beta 20, 20)
alpha_strong_prior, beta_strong_prior = 20, 20
alpha_strong_post = alpha_strong_prior + recoveries
beta_strong_post = beta_strong_prior + failures

print('Data pushes the weak prior heavily, but struggles to move the strong prior.')
print(f'Weak Prior Posterior Mean: {alpha_weak_post / (alpha_weak_post + beta_weak_post):.3f}')
print(f'Strong Prior Posterior Mean: {alpha_strong_post / (alpha_strong_post + beta_strong_post):.3f}')

## 10. Visualizing the Effect of Strong vs Weak Priors

Plotting the posteriors derived from different priors demonstrates how Bayesian inference automatically balances our initial confidence against the weight of new evidence.

In [ ]:
pdf_weak_post = stats.beta.pdf(x_vals, alpha_weak_post, beta_weak_post)
pdf_strong_prior = stats.beta.pdf(x_vals, alpha_strong_prior, beta_strong_prior)
pdf_strong_post = stats.beta.pdf(x_vals, alpha_strong_post, beta_strong_post)

plt.figure(figsize=(10, 6))
plt.plot(x_vals, pdf_weak_post, 'b-', lw=3, label='Posterior from Weak Prior')
plt.plot(x_vals, pdf_strong_prior, 'r--', lw=2, label='Strong Prior: Beta(20,20)')
plt.plot(x_vals, pdf_strong_post, 'r-', lw=3, label='Posterior from Strong Prior')

# True parameter line
plt.axvline(true_efficacy, color='green', linestyle=':', lw=2, label=f'True Efficacy ({true_efficacy})')

plt.title('How Priors Influence the Posterior', fontsize=14)
plt.xlabel('Efficacy Probability (theta)')
plt.ylabel('Density')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 11. Core Concept 4: Bayesian vs Frequentist Intervals

Frequentist Confidence Interval: 'If we repeat the experiment infinite times, 95 percent of calculated intervals will contain the true parameter.'

Bayesian Credible Interval (Highest Density Interval): 'Given this data and my prior, there is a 95 percent probability that the true parameter lies in this specific interval.' This is what most business stakeholders actually want to know.

In [ ]:
# 1. Frequentist 95% Confidence Interval (Normal Approximation)
p_hat = recoveries / n_patients
z_score = 1.96
margin_of_error = z_score * np.sqrt((p_hat * (1 - p_hat)) / n_patients)
freq_ci_lower = p_hat - margin_of_error
freq_ci_upper = p_hat + margin_of_error

# 2. Bayesian 95% Credible Interval (using Percentile Point Function / Inverse CDF)
# We use the weak posterior for comparison
bayes_ci_lower = stats.beta.ppf(0.025, alpha_weak_post, beta_weak_post)
bayes_ci_upper = stats.beta.ppf(0.975, alpha_weak_post, beta_weak_post)

print('--- Uncertainty Intervals Comparison ---')
print(f'Frequentist 95% CI:       [{freq_ci_lower:.3f}, {freq_ci_upper:.3f}]')
print(f'Bayesian 95% Credible Int: [{bayes_ci_lower:.3f}, {bayes_ci_upper:.3f}]')
print('\nNotice they are similar with flat priors, but the Bayesian interpretation is fundamentally different.')

## 12. Practical Example: A/B Testing in Marketing

Let us apply Bayesian Inference to a classic business problem: A/B Testing. We are testing two website designs to see which yields a higher conversion rate. We will simulate conversions for Variant A and Variant B.

In [ ]:
# Generate A/B Test Data
n_visitors = 1000
true_rate_A = 0.040  # 4% conversion
true_rate_B = 0.052  # 5.2% conversion

conversions_A = np.random.binomial(n_visitors, true_rate_A)
conversions_B = np.random.binomial(n_visitors, true_rate_B)

print('A/B Test Raw Results:')
print(f'Variant A: {conversions_A} conversions out of {n_visitors} ({conversions_A/n_visitors*100:.1f}%)')
print(f'Variant B: {conversions_B} conversions out of {n_visitors} ({conversions_B/n_visitors*100:.1f}%)')

## 13. Bayesian Analysis of the A/B Test

Instead of calculating a p-value, we will calculate the Posterior distributions for both A and B. Then, we can answer the direct business question: 'What is the probability that Variant B is better than Variant A?'

We will answer this by drawing thousands of random samples from both Posterior distributions and comparing them.

In [ ]:
# Define Priors (Beta(1,1) for uniform ignorance)
a_prior, b_prior = 1, 1

# Calculate Posteriors
a_post_A = a_prior + conversions_A
b_post_A = b_prior + (n_visitors - conversions_A)

a_post_B = a_prior + conversions_B
b_post_B = b_prior + (n_visitors - conversions_B)

# Monte Carlo Simulation: Draw 100,000 samples from each posterior
n_samples = 100000
samples_A = np.random.beta(a_post_A, b_post_A, n_samples)
samples_B = np.random.beta(a_post_B, b_post_B, n_samples)

# Calculate Probability that B is better than A
prob_B_better = np.mean(samples_B > samples_A)

# Calculate the Expected Uplift
expected_uplift = np.mean((samples_B - samples_A) / samples_A) * 100

print(f'Bayesian A/B Test Results:')
print(f'Probability that B is better than A: {prob_B_better * 100:.2f}%')
print(f'Expected Relative Uplift: {expected_uplift:.2f}%')
print('\nThis is a highly actionable statement for a product manager!')

## 14. Visualizing the A/B Test Posteriors

We can plot the two Beta distributions to see the overlap. The less overlap, the more confident we are that the variants are truly different.

In [ ]:
x_ab = np.linspace(0.01, 0.08, 1000)
pdf_A = stats.beta.pdf(x_ab, a_post_A, b_post_A)
pdf_B = stats.beta.pdf(x_ab, a_post_B, b_post_B)

plt.figure(figsize=(10, 6))
plt.plot(x_ab, pdf_A, label='Variant A Posterior', color='red', lw=2)
plt.fill_between(x_ab, 0, pdf_A, alpha=0.2, color='red')

plt.plot(x_ab, pdf_B, label='Variant B Posterior', color='blue', lw=2)
plt.fill_between(x_ab, 0, pdf_B, alpha=0.2, color='blue')

plt.title('A/B Test Posterior Distributions', fontsize=14)
plt.xlabel('Conversion Rate')
plt.ylabel('Density')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 15. Common Pitfalls: The Zero Prior

A massive trap in Bayesian inference is assigning a 0 probability to an event in your Prior. If your Prior is 0, Bayes Theorem multiplies the Likelihood by 0. No amount of evidence can ever change your mind. Let us demonstrate this mathematically.

In [ ]:
# Imagine a rigid prior that absolutely refuses to believe efficacy can be above 0.5
rigid_prior = np.where(p_grid <= 0.5, 2.0, 0.0)
rigid_prior = rigid_prior / np.sum(rigid_prior) # Normalize

# New overwhelming evidence: 50 successes out of 50 trials!
overwhelming_likelihood = stats.binom.pmf(k=50, n=50, p=p_grid)

# Update
rigid_unnorm_post = overwhelming_likelihood * rigid_prior
rigid_posterior = rigid_unnorm_post / np.sum(rigid_unnorm_post)

print(f'Max Posterior Probability despite 50/50 successes: p = {p_grid[np.argmax(rigid_posterior)]:.2f}')
print('Lesson: Never assign a 0 probability to an event unless it is physically impossible.')

## 16. Practice Exercises

Scenario: A manufacturing plant produces microchips. The defect rate is historically known to hover around 2 percent, acting as our prior (Beta with alpha=2, beta=98). Today, a new batch of 500 chips is tested, and 15 are defective.

Your Task:
1. Calculate the parameters for the Posterior Beta distribution.
2. Determine the Expected Defect Rate (the mean of the posterior).

In [ ]:
# Exercise Data Setup
hist_alpha, hist_beta = 2, 98
chips_tested = 500
defects_found = 15

print(f'Historical Prior: Beta({hist_alpha}, {hist_beta})')
print(f'Test Data: {defects_found} defects out of {chips_tested} chips.')

## 17. Exercise Solution

Using the conjugate prior rules for Beta-Binomial:
New Alpha = Prior Alpha + Successes (Defects in this context)
New Beta = Prior Beta + Failures (Non-defects)

In [ ]:
# Solution Calculation
post_alpha_ex = hist_alpha + defects_found
post_beta_ex = hist_beta + (chips_tested - defects_found)

expected_defect_rate = post_alpha_ex / (post_alpha_ex + post_beta_ex)

print('--- Exercise Solution ---')
print(f'1. Posterior Parameters: Beta({post_alpha_ex}, {post_beta_ex})')
print(f'2. Expected Defect Rate (Posterior Mean): {expected_defect_rate * 100:.2f}%')
print('Notice how the historical prior tempered the unusually high 3% defect rate seen in today\'s batch.')

## 18. Visualization Gallery: Sequential Updating

The beauty of Bayes is that today's Posterior becomes tomorrow's Prior. Let us visualize data arriving sequentially, updating our belief step-by-step.

In [ ]:
# True rate is 0.6. Data arrives in batches of 10.
batches = [ 
    {'success': 6, 'total': 10},
    {'success': 7, 'total': 10},
    {'success': 5, 'total': 10},
    {'success': 6, 'total': 10}
]

plt.figure(figsize=(12, 6))
x_seq = np.linspace(0, 1, 500)
current_alpha, current_beta = 1, 1 # Start Uniform

# Plot initial prior
plt.plot(x_seq, stats.beta.pdf(x_seq, current_alpha, current_beta), 'k--', label='Initial Prior', alpha=0.5)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for i, batch in enumerate(batches):
    current_alpha += batch['success']
    current_beta += (batch['total'] - batch['success'])
    
    y_seq = stats.beta.pdf(x_seq, current_alpha, current_beta)
    plt.plot(x_seq, y_seq, color=colors[i], lw=2, label=f'After Batch {i+1} (N={current_alpha+current_beta-2})')
    plt.fill_between(x_seq, 0, y_seq, color=colors[i], alpha=0.1)

plt.axvline(0.6, color='black', linestyle=':', lw=2, label='True Parameter (0.6)')
plt.title('Sequential Bayesian Updating', fontsize=14)
plt.xlabel('Parameter Value')
plt.ylabel('Density')
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print('With every batch of data, the distribution becomes narrower (more confident) and hones in on the truth.')

## 19. Summary and Key Takeaways

- Uncertainty as Belief: Bayesian inference treats parameters as random variables with probability distributions, unlike the Frequentist view of fixed constants.
- Bayes Theorem Framework: Posterior is proportional to Likelihood * Prior. We use data to update our initial assumptions.
- Conjugate Priors: Choosing matching distributions (like Beta and Binomial) allows for exact, lightning-fast analytical solutions via simple addition.
- Actionable Insights: Instead of binary p-values, Bayesian methods allow us to ask direct business questions, like 'What is the exact probability that Variant B is better than Variant A?'
- Pitfalls: Overly rigid priors (especially assigning 0 probability) can prevent the model from learning the truth, no matter how much data is provided.